In [0]:
import time
from pyspark.ml.evaluation import BinaryClassificationEvaluator


def run_baseline(df):
    from pyspark.ml.classification import LogisticRegression
    from pyspark.ml.feature import VectorAssembler

    features = [c for c in df.columns if c not in ["label", "weight"]]

    assembler = VectorAssembler(inputCols=features, outputCol="features")
    data = assembler.transform(df)

    train, test = data.randomSplit([0.8, 0.2], seed=42)

    model = LogisticRegression(labelCol="label", featuresCol="features")
    fitted = model.fit(train)
    preds = fitted.transform(test)

    auc = BinaryClassificationEvaluator().evaluate(preds)

    print("\n=== BASELINE ===")
    print("AUC:", auc)

    return auc


# ===========================================================
# RANDOM SEARCH BASELINE
# ===========================================================
# Fair comparison with GA requires:
#   1. Same total number of model evaluations  (n_evals = pop_size * generations)
#   2. Same search space                       (uses create_chromosome from 01_ga_core)
#   3. Same train/test split and fitness fn    (uses evaluate_population from 02_spark_engine)
#
# Random Search has NO memory — each candidate is independently sampled.
# It is the standard baseline to prove that GA's directed search adds value.
# ===========================================================

def run_random_search(eval_fn, n_evals, batch_size=6):
    """
    Randomly samples chromosomes from SEARCH_SPACE in batches.
    Evaluates them in the same parallel batches as the GA uses,
    so wall-clock time is directly comparable.

    Parameters
    ----------
    eval_fn    : same evaluate_population function used by the GA
    n_evals    : total number of models to evaluate
                 (set equal to pop_size * generations for a fair GA comparison)
    batch_size : how many candidates to evaluate in parallel per round
                 (set equal to GA pop_size)

    Returns
    -------
    best_chromosome : (model_type, params) with highest fitness found
    best_fitness    : float
    history         : list of best-so-far fitness after each batch
                      (same structure as GA history for direct plot comparison)
    """
    print(f"\n=== RANDOM SEARCH: {n_evals} total evaluations in batches of {batch_size} ===")

    best_fitness    = -1
    best_chromosome = None
    history         = []   # best-so-far after each batch
    all_results     = []

    evaluated = 0
    batch_num = 0

    while evaluated < n_evals:
        # Sample a fresh random batch — no memory of previous results
        this_batch = min(batch_size, n_evals - evaluated)
        batch = [create_chromosome() for _ in range(this_batch)]

        batch_num += 1
        print(f"\n--- Random Search Batch {batch_num} ({evaluated + this_batch}/{n_evals} evaluated) ---")

        results = eval_fn(batch)
        all_results.extend(results)

        # Update best found so far
        for fitness, chrom in results:
            if fitness > best_fitness:
                best_fitness    = fitness
                best_chromosome = chrom

        history.append(best_fitness)
        evaluated += this_batch

        print(f"Best fitness so far: {best_fitness:.4f}")

    print(f"\n=== RANDOM SEARCH COMPLETE ===")
    print(f"Best Model  : {best_chromosome[0]}")
    print(f"Best Params : {best_chromosome[1]}")
    print(f"Best Fitness: {best_fitness:.4f}")

    return best_chromosome, best_fitness, history


def scalability_test(df, fn):
    results = []
    for frac in [0.1, 0.3, 0.5, 0.8]:
        # FIX: df.sample() requires withReplacement as first positional arg
        sample = df.sample(withReplacement=False, fraction=frac, seed=42)

        start = time.time()
        fn(sample)
        duration = time.time() - start

        print(f"Fraction {frac} -> {duration:.2f} sec")
        results.append((frac, duration))

    return results


def parallelism_test(df, fn):
    results = []
    for p in [2, 4, 8, 16]:
        dfp = df.repartition(p)

        start = time.time()
        fn(dfp)
        duration = time.time() - start

        print(f"Partitions {p} -> {duration:.2f} sec")
        results.append((p, duration))

    return results